<a href="https://colab.research.google.com/github/rashidulhaq-coding/Machine-learning-projects/blob/master/News_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install datasets
# !pip install gradio

In [ ]:
import torch
import gradio as gr
from datasets import Dataset,load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
dataset = load_dataset("gopalkalpande/bbc-news-summary")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['File_path', 'Articles', 'Summaries'],
        num_rows: 2224
    })
})

In [ ]:

model_checkpoint = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_input_length = 1024
max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(examples["Articles"], max_length=max_input_length, truncation=True)
    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["Summaries"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

batch_size = 8
args = Seq2SeqTrainingArguments(
    "test-summarization",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    push_to_hub=False,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


In [ ]:

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

<ipython-input-7-7e94da824843>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=278, training_loss=0.18009978918720493, metrics={'train_runtime': 859.3448, 'train_samples_per_second': 2.588, 'train_steps_per_second': 0.324, 'total_flos': 3795158864953344.0, 'train_loss': 0.18009978918720493, 'epoch': 1.0})

In [ ]:
article = """
US President Donald Trump has called Canadian Prime Minister Mark Carney to congratulate him on his victory in the country's general election and the two have agreed to meet in the near future.

The two countries were expected to enter talks about a new economic and security relationship after Monday's vote.

Trump's trade tariffs and repeated comments undermining Canada's sovereignty overshadowed the race, which ended with Carney's Liberals projected to win a minority government, according to public broadcaster CBC.

That result will make Carney's pressing tasks of negotiating with his US counterpart and tackling a range of domestic issues more of a challenge, as he will need to wrangle support from other political parties.

In their first call since the election, Trump congratulated Carney on his victory, according to the prime minister's office on Tuesday.

The office also said the two leaders had "agreed on the importance of Canada and the United States working together – as independent, sovereign nations – for their mutual betterment".

The Liberals will need to rely on their support to pass legislation through the House of Commons.

They also face possible defeat in any vote of confidence in the chamber.
"""

def summarize(article):
    inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True)

    # Move input tensors to the same device as the model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=128,
        num_beams=4,
        early_stopping=True,
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
summarize(article)

'The office also said the two leaders had "agreed on the importance of Canada and the United States working together – as independent, sovereign nations – for their mutual betterment".In their first call since the election, Trump congratulated Carney on his victory, according to the prime minister\'s office on Tuesday.'

In [ ]:

iface = gr.Interface(
    fn=summarize,
    inputs=gr.Textbox(lines=15, label="Input Article"),
    outputs=gr.Textbox(label="Generated Summary"),
    title="Article Summarizer",
    description="Paste an article and get a summary using your fine-tuned BART model.",
)

iface.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6edaaf15ae42235328.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
